# Combine brakes and NFL separation

Kaggle notebook skeleton for NFL Big Data Bowl 2027. The competition entry is this notebook plus a write-up. It is not a prediction CSV.

The notebook reads the nine official CSVs from `/kaggle/input/nfl-big-data-bowl-2027/`. It does not download them. Accept the competition rules in the Kaggle UI before the files are available.

Pre-registered questions:

- Tight ends: do 10 Hz braking on TE position drills, the 10-yard split, and the 3-cone line up with separation and target outcomes inside a draft class?
- Defensive backs: does the backpedal-to-break drill line up with separation, completions, and EPA allowed?

Details and the rules checklist live in the project docs. The external rating comparison is off.


## Setup

Install this repository on the Kaggle notebook if `bdb2027` is not already importable. The next cell adds a local `src/` directory when the notebook is run from a checkout.


In [ ]:
import sys
from pathlib import Path

for candidate in (Path.cwd() / "src", Path.cwd(), Path("/kaggle/working/src")):
    if (candidate / "bdb2027").is_dir():
        sys.path.insert(0, str(candidate))
        break


In [ ]:
from pathlib import Path

from bdb2027.io import default_data_dir, missing_official_files
from bdb2027.pipeline import effects_summary, run_analysis
from bdb2027.preregistration import PREREGISTERED_TESTS, bonferroni_alpha

DATA_DIR = Path("/kaggle/input/nfl-big-data-bowl-2027")
if not DATA_DIR.is_dir():
    DATA_DIR = default_data_dir()

missing = missing_official_files(DATA_DIR)
if missing:
    raise FileNotFoundError(
        "Competition CSVs are not available at "
        f"{DATA_DIR}. Missing: {', '.join(missing)}. "
        "Accept the Kaggle rules and add the competition data. Do not commit the files."
    )
print("Data directory:", DATA_DIR)
print("Pre-registered tests:", len(PREREGISTERED_TESTS))
print("Bonferroni alpha:", round(bonferroni_alpha(), 4))


## Pre-registered effects

`run_analysis` loads only the columns it needs, in chunks. Combine features use player rows from the TE skill drills, the 3-cone, and the backpedal drill. DB coverage outcomes use `pass_forward` frames only.

The table below is the within-draft-class Pearson correlation, with a 95% Fisher interval. `small_n` is true when fewer than 20 players enter the test.


In [ ]:
result = run_analysis(DATA_DIR)
effects = result["effects"]
print(effects_summary(effects))
effects


## Charts to embed in the write-up

Keep the finished write-up under 2,000 words and under 10 tables or figures. The cells below are the two primary scatters. They plot within-class deviations, so a year effect is not the slope.


In [ ]:
import matplotlib.pyplot as plt


def within_class_deviation(frame, column, class_col="draft_year"):
    values = frame[column]
    return values - values.groupby(frame[class_col]).transform("mean")


def scatter_deviations(frame, predictor, outcome, title):
    plot = frame.dropna(subset=[predictor, outcome, "draft_year"]).copy()
    if plot.empty:
        print("No rows for", title)
        return
    plt.figure(figsize=(6, 4))
    plt.scatter(
        within_class_deviation(plot, predictor),
        within_class_deviation(plot, outcome),
        alpha=0.8,
    )
    plt.axhline(0, color="0.5", linewidth=0.8)
    plt.axvline(0, color="0.5", linewidth=0.8)
    plt.xlabel(predictor + " (within draft class)")
    plt.ylabel(outcome + " (within draft class)")
    plt.title(title)
    plt.tight_layout()
    plt.show()


scatter_deviations(
    result["te_players"],
    "te_skill_decel_magnitude",
    "te_mean_separation",
    "TE primary: drill brake and separation",
)
scatter_deviations(
    result["db_players"],
    "db_break_time_s",
    "db_mean_separation_allowed",
    "DB primary: time to break and separation allowed",
)


## Optional external benchmark

Leave this cell unchanged unless the rating file is publicly available to every participant at no cost, or it meets the host Reasonableness Standard. Madden ratings were not verified to meet that test. Do not scrape ratings and do not commit them.

Set `ENABLE_EXTERNAL_BENCHMARK` to `True` and point `RATING_CSV` at a local file with `nfl_id` and any of: `speed`, `acceleration`, `agility`, `change_of_direction`, `route_running`, `release`, `man_coverage`, `zone_coverage`.


In [ ]:
ENABLE_EXTERNAL_BENCHMARK = False
RATING_CSV = None  # path outside the repo and outside the competition CSVs

external = run_analysis(
    DATA_DIR,
    external_benchmark_path=None if RATING_CSV is None else Path(RATING_CSV),
    external_benchmark_enabled=ENABLE_EXTERNAL_BENCHMARK,
)["external"]
print(external["status"])
print(external["reason"])
if external["status"] == "computed":
    display(external["comparisons"])


## Sample sizes

Replace the planning notes in the write-up with these counts. Do not treat a planning guess of about 18 tight ends or about 62 defensive backs as a measured sample.


In [ ]:
print("Games loaded:", result["n_games"])
print("TE players in the merge:", len(result["te_players"]))
print("DB players in the merge:", len(result["db_players"]))
print(effects.loc[:, ["test_id", "n_players", "n_classes", "class_counts", "small_n"]])
